# Stage 1 Ablation — Hungarian Assignment ON vs OFF

**What this tests:** the v4 run (`../v3-output/`) got **100% success, 0% collision on every single one of 50 checkpoints** across 5000 episodes. That is suspicious — zero variance for an entire run usually means the task configuration is too easy to prove anything, not that the method is great.

The one thing we have NOT tested: does the **Hungarian per-step target assignment** (the mechanism this thesis line of work depends on) actually matter here, or would *any* fixed pairing of drones to targets do just as well because the world is huge, obstacle-free, and the time budget is generous?

**This notebook is identical to v4 in every other way** (same reward, same `target_radius=25`, `success_bonus=20`, `ent_coef=0.003`, `total_episodes=5000`, same seed) — the **only** change is one flag: `use_hungarian`.

| Condition | Assignment |
|---|---|
| `use_hungarian=True` (already run, see `../v3-output/`) | Optimal, re-solved every step (`scipy.optimize.linear_sum_assignment`) |
| `use_hungarian=False` (this notebook) | **Fixed at reset**: drone `i` → target `i`, arbitrary, never re-solved |

**How to read the result:**
- If success **drops noticeably** below 100% without Hungarian assignment → good, the mechanism matters, Stage 1 as configured is a valid test.
- If success **stays ~100%** without it too → Stage 1 is too easy to discriminate the mechanism. Before Stage 2, tighten `target_radius` and/or `max_steps` and/or add obstacles so the task actually requires good assignment.

One variable changed, nothing else — so the result is directly attributable.

In [ ]:
# Cell 1 — Imports
import torch
import numpy as np
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device    : {device}')


In [ ]:
# Cell 2 — 2D Environment (identical to v4, + one ablation flag)
#
# Everything below is unchanged from v4 EXCEPT the `use_hungarian` flag:
#   use_hungarian=True  -> reset() and step() both call _hungarian() every step (v4 behaviour)
#   use_hungarian=False -> assignment is fixed at reset (drone i -> target i, arbitrary
#                           index pairing, never re-solved) — this is the ablation.
#
# Everything else (reward formula, normalization, success bonus, graded r_safety) is
# byte-for-byte the same as v4, so any difference in the result is attributable to the
# assignment mechanism alone.

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from scipy.optimize import linear_sum_assignment

class MultiUAVEnv(gym.Env):
    def __init__(self, n_drones=3, n_obstacles=0, world_size=500.0,
                 max_speed=5.0, max_steps=300, collision_radius=3.0,
                 target_radius=25.0, success_bonus=20.0,
                 use_hungarian=True, seed=None):
        super().__init__()
        self.n_drones         = n_drones
        self.n_obstacles      = n_obstacles
        self.world_size       = world_size
        self.max_speed        = max_speed
        self.max_steps        = max_steps
        self.collision_radius = collision_radius
        self.target_radius    = target_radius
        self.success_bonus    = success_bonus
        self.use_hungarian    = use_hungarian   # <-- the ablation switch
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(n_drones, 10), dtype=np.float32)
        self.action_space = spaces.Box(
            low=-max_speed, high=max_speed, shape=(n_drones, 2), dtype=np.float32)
        self.rng          = np.random.default_rng(seed)
        self.drone_pos    = self.drone_vel = self.target_pos = None
        self.obstacle_pos = np.zeros((0, 2), dtype=np.float32)
        self.assignment   = None
        self.step_count   = 0

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.step_count   = 0
        n_total           = self.n_drones * 2 + self.n_obstacles
        all_pos           = self._sample_non_overlapping(n_total, self.collision_radius * 2)
        self.drone_pos    = all_pos[:self.n_drones].copy()
        self.target_pos   = all_pos[self.n_drones:2*self.n_drones].copy()
        self.obstacle_pos = (all_pos[2*self.n_drones:].copy()
                             if self.n_obstacles > 0
                             else np.zeros((0, 2), dtype=np.float32))
        self.drone_vel    = np.zeros((self.n_drones, 2), dtype=np.float32)
        # Ablation point: fixed identity pairing when Hungarian is off. Positions are
        # sampled in arbitrary order, so drone i -> target i is an arbitrary (usually
        # non-optimal) pairing, not a disguised optimal one.
        self.assignment   = self._hungarian() if self.use_hungarian else np.arange(self.n_drones)
        return self._obs(), {}

    def step(self, actions):
        self.step_count  += 1
        self.drone_vel    = np.clip(actions, -self.max_speed, self.max_speed).astype(np.float32)
        self.drone_pos    = np.clip(self.drone_pos + self.drone_vel, 0.0, self.world_size)
        if self.use_hungarian:
            self.assignment = self._hungarian()
        # else: assignment stays exactly as set at reset() — no re-solving, no optimization.
        rewards           = self._rewards()
        reached           = self._all_reached()
        collision         = self._any_collision()

        if reached:
            rewards += self.success_bonus

        terminated = reached or collision
        truncated  = self.step_count >= self.max_steps
        info = {
            'all_targets_reached': reached,
            'any_collision':       collision,
            'timeout':             truncated,
            'step':                self.step_count,
        }
        return self._obs(), rewards, terminated, truncated, info

    def _obs(self):
        obs = np.zeros((self.n_drones, 10), dtype=np.float32)
        for i in range(self.n_drones):
            rel = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            obs[i] = np.concatenate([
                self.drone_pos[i] / self.world_size,  # [0, 1]
                self.drone_vel[i] / self.max_speed,   # [-1, 1]
                rel               / self.world_size,  # [-1, 1]
                self._clearances(i),                  # [0, 1]
            ])
        return obs

    def _rewards(self):
        rewards    = np.zeros(self.n_drones, dtype=np.float32)
        d_danger   = self.collision_radius * 3.0
        zone_width = d_danger - self.collision_radius

        for i in range(self.n_drones):
            rel_target = self.target_pos[self.assignment[i]] - self.drone_pos[i]
            dist       = np.linalg.norm(rel_target)

            r_mission = -dist / self.world_size

            direction  = rel_target / (dist + 1e-6)
            r_progress = float(np.dot(self.drone_vel[i], direction) / self.max_speed)

            min_dist = float('inf')
            for j in range(self.n_drones):
                if j == i: continue
                min_dist = min(min_dist, np.linalg.norm(self.drone_pos[i] - self.drone_pos[j]))
            for obs_pos in self.obstacle_pos:
                min_dist = min(min_dist, np.linalg.norm(self.drone_pos[i] - obs_pos))

            if min_dist < self.collision_radius:
                r_safety = -1.0
            elif min_dist < d_danger:
                r_safety = -((d_danger - min_dist) / zone_width)
            else:
                r_safety = 0.0

            rewards[i] = 0.4 * r_progress + 0.3 * r_mission + 0.3 * r_safety
        return rewards

    def _hungarian(self):
        cost = np.linalg.norm(self.drone_pos[:, None] - self.target_pos[None, :], axis=-1)
        _, cols = linear_sum_assignment(cost)
        return cols

    def _drone_col(self, i):
        dists    = np.linalg.norm(self.drone_pos[i] - self.drone_pos, axis=-1)
        dists[i] = np.inf
        return bool(np.any(dists < self.collision_radius))

    def _obs_col(self, i):
        if len(self.obstacle_pos) == 0:
            return False
        return bool(np.any(
            np.linalg.norm(self.drone_pos[i] - self.obstacle_pos, axis=-1) < self.collision_radius
        ))

    def _any_collision(self):
        return any(self._drone_col(i) or self._obs_col(i) for i in range(self.n_drones))

    def _all_reached(self):
        return all(
            np.linalg.norm(self.drone_pos[i] - self.target_pos[self.assignment[i]]) <= self.target_radius
            for i in range(self.n_drones)
        )

    def _clearances(self, i):
        pos = self.drone_pos[i]
        c   = np.array([
            self.world_size - pos[1],
            pos[1],
            self.world_size - pos[0],
            pos[0],
        ], dtype=np.float32)
        return c / self.world_size

    def _sample_non_overlapping(self, n, min_dist):
        positions, attempts = [], 0
        while len(positions) < n:
            attempts += 1
            if attempts > 10000:
                raise RuntimeError('World too crowded')
            p = self.rng.uniform(10.0, self.world_size - 10.0, size=2)
            if not any(np.linalg.norm(p - q) < min_dist for q in positions):
                positions.append(p)
        return np.array(positions, dtype=np.float32)

print('Environment ready! (ablation build — use_hungarian flag added, everything else = v4)')


In [ ]:
# Cell 3 — MAPPO (Actor + Critic + Buffer) — unchanged from v4, no PAH (Stage 1)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
            nn.Linear(hidden, action_dim),
        )
        self.log_std = nn.Parameter(torch.zeros(action_dim))
    def _dist(self, obs):
        return Normal(self.net(obs), self.log_std.exp().expand_as(self.net(obs)))
    def get_action(self, obs):
        d = self._dist(obs); a = d.sample()
        return a, d.log_prob(a).sum(-1)
    def evaluate_action(self, obs, action):
        d = self._dist(obs)
        return d.log_prob(action).sum(-1), d.entropy().sum(-1)

class Critic(nn.Module):
    def __init__(self, global_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(global_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),     nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

class RolloutBuffer:
    def __init__(self, n_drones):
        self.n_drones = n_drones
        self.clear()
    def clear(self):
        self.obs=[]; self.actions=[]; self.rewards=[]
        self.values=[]; self.log_probs=[]; self.dones=[]
    def add(self, obs, actions, rewards, value, log_probs, done):
        self.obs.append(obs.copy()); self.actions.append(actions.copy())
        self.rewards.append(rewards.copy()); self.values.append(float(value))
        self.log_probs.append(log_probs.copy()); self.dones.append(bool(done))
    def __len__(self): return len(self.rewards)
    def compute_gae(self, last_value, gamma=0.99, lam=0.95):
        T = len(self.rewards); n = self.n_drones
        rewards = np.array(self.rewards, dtype=np.float32)
        values  = np.array(self.values,  dtype=np.float32)
        dones   = np.array(self.dones,   dtype=np.float32)
        adv = np.zeros((T, n), dtype=np.float32)
        last_gae = np.zeros(n, dtype=np.float32)
        for t in reversed(range(T)):
            nv    = last_value if t == T-1 else values[t+1]
            m     = 1.0 - dones[t]
            delta = rewards[t] + gamma * nv * m - values[t]
            last_gae = delta + gamma * lam * m * last_gae
            adv[t] = last_gae
        return adv, adv + values[:, np.newaxis]
    def to_tensors(self, device):
        obs = torch.tensor(np.array(self.obs),       dtype=torch.float32, device=device)
        act = torch.tensor(np.array(self.actions),   dtype=torch.float32, device=device)
        lp  = torch.tensor(np.array(self.log_probs), dtype=torch.float32, device=device)
        return obs, act, lp

class MAPPO:
    def __init__(self, n_drones, obs_dim, action_dim, max_speed,
                 lr=3e-4, gamma=0.99, lam=0.95, clip_eps=0.2,
                 vf_coef=0.5, ent_coef=0.003, n_epochs=10, batch_size=64):
        self.n_drones=n_drones; self.max_speed=max_speed
        self.gamma=gamma; self.lam=lam; self.clip_eps=clip_eps
        self.vf_coef=vf_coef; self.ent_coef=ent_coef
        self.n_epochs=n_epochs; self.batch_size=batch_size
        self.device = DEVICE
        self.actor  = Actor(obs_dim, action_dim).to(DEVICE)
        self.critic = Critic(n_drones * obs_dim).to(DEVICE)
        self.opt    = optim.Adam(
            list(self.actor.parameters()) + list(self.critic.parameters()), lr=lr)
    @torch.no_grad()
    def get_actions(self, obs_np):
        obs_t = torch.tensor(obs_np, dtype=torch.float32, device=self.device)
        a, lp = self.actor.get_action(obs_t)
        a     = a.clamp(-self.max_speed, self.max_speed)
        v     = self.critic(obs_t.flatten().unsqueeze(0)).item()
        return a.cpu().numpy(), lp.cpu().numpy(), v
    def update(self, buffer, last_obs_np):
        lo = torch.tensor(last_obs_np, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            lv = self.critic(lo.flatten().unsqueeze(0)).item()
        adv_np, ret_np = buffer.compute_gae(lv, self.gamma, self.lam)
        obs_t, act_t, olp_t = buffer.to_tensors(self.device)
        T, n, _ = obs_t.shape
        adv_t   = torch.tensor(adv_np, dtype=torch.float32, device=self.device)
        ret_t   = torch.tensor(ret_np, dtype=torch.float32, device=self.device)
        af      = adv_t.reshape(-1)
        adv_t   = (adv_t - af.mean()) / (af.std() + 1e-8)
        obs_f   = obs_t.reshape(T*n, -1)
        act_f   = act_t.reshape(T*n, -1)
        olp_f   = olp_t.reshape(T*n)
        adv_f   = adv_t.reshape(T*n)
        ret_f   = ret_t.reshape(T*n)
        gs      = obs_t.reshape(T, -1).unsqueeze(1).expand(T, n, -1).reshape(T*n, -1)
        idx_all = np.arange(T*n)
        al, cl, el = [], [], []
        for _ in range(self.n_epochs):
            np.random.shuffle(idx_all)
            for s in range(0, T*n, self.batch_size):
                idx    = idx_all[s:s+self.batch_size]
                lp, ent = self.actor.evaluate_action(obs_f[idx], act_f[idx])
                r      = (lp - olp_f[idx]).exp()
                loss_a = -torch.min(
                    r * adv_f[idx],
                    r.clamp(1-self.clip_eps, 1+self.clip_eps) * adv_f[idx]
                ).mean()
                loss_c = 0.5 * (self.critic(gs[idx]) - ret_f[idx]).pow(2).mean()
                loss   = loss_a + self.vf_coef * loss_c - self.ent_coef * ent.mean()
                self.opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(
                    list(self.actor.parameters()) + list(self.critic.parameters()), 0.5)
                self.opt.step()
                al.append(loss_a.item()); cl.append(loss_c.item()); el.append(ent.mean().item())
        buffer.clear()
        return {'actor_loss': np.mean(al), 'critic_loss': np.mean(cl), 'entropy': np.mean(el)}
    def save(self, path):
        torch.save({'actor': self.actor.state_dict(), 'critic': self.critic.state_dict()}, path)

print('MAPPO ready!')


In [ ]:
# Cell 4 — Training setup
import os, time, json

# Identical to v4 CFG except: use_hungarian=False, run_name changed.
# This is the ONLY intended difference for this run — keep it that way.
CFG = {
    'n_drones':        3,
    'n_obstacles':     0,
    'world_size':      500.0,
    'max_speed':       5.0,
    'max_steps':       300,
    'target_radius':   25.0,
    'success_bonus':   20.0,
    'use_hungarian':   False,     # <-- THE ABLATION. Everything else matches v4.
    'total_episodes':  5000,
    'rollout_steps':   512,
    'eval_every':      100,
    'eval_episodes':   20,
    'save_every':      500,
    'seed':            42,
    'lr':              3e-4,
    'gamma':           0.99,
    'lam':             0.95,
    'clip_eps':        0.2,
    'vf_coef':         0.5,
    'ent_coef':        0.003,
    'n_epochs':        10,
    'batch_size':      64,
    'run_name':        'output-ablation-no-hungarian',
}

np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CFG['seed'])

train_env = MultiUAVEnv(
    n_drones=CFG['n_drones'],           n_obstacles=CFG['n_obstacles'],
    world_size=CFG['world_size'],       max_speed=CFG['max_speed'],
    max_steps=CFG['max_steps'],         target_radius=CFG['target_radius'],
    success_bonus=CFG['success_bonus'], use_hungarian=CFG['use_hungarian'],
    seed=CFG['seed']
)
eval_env = MultiUAVEnv(
    n_drones=CFG['n_drones'],           n_obstacles=CFG['n_obstacles'],
    world_size=CFG['world_size'],       max_speed=CFG['max_speed'],
    max_steps=CFG['max_steps'],         target_radius=CFG['target_radius'],
    success_bonus=0.0,                  use_hungarian=CFG['use_hungarian'],
    seed=CFG['seed'] + 999
)

OBS_DIM    = train_env.observation_space.shape[1]   # 10
ACTION_DIM = train_env.action_space.shape[1]         # 2

agent  = MAPPO(
    CFG['n_drones'], OBS_DIM, ACTION_DIM, CFG['max_speed'],
    lr=CFG['lr'], gamma=CFG['gamma'], lam=CFG['lam'],
    clip_eps=CFG['clip_eps'], vf_coef=CFG['vf_coef'],
    ent_coef=CFG['ent_coef'], n_epochs=CFG['n_epochs'],
    batch_size=CFG['batch_size']
)
buffer = RolloutBuffer(CFG['n_drones'])

RESULTS_DIR = os.path.join(os.getcwd(), CFG['run_name'])
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Device          : {DEVICE}")
print(f"World size      : {CFG['world_size']} m x {CFG['world_size']} m")
print(f"Target radius   : {CFG['target_radius']} m")
print(f"Use Hungarian   : {CFG['use_hungarian']}  <-- ablation flag")
print(f"Results dir     : {RESULTS_DIR}")
print('Setup complete! Ready to train.')


In [ ]:
# Cell 5 — TRAIN  (~30-45 minutes, same budget as v4)

def collect_rollout(env, agent, buffer, steps):
    obs, _ = env.reset()
    for _ in range(steps):
        actions, log_probs, value = agent.get_actions(obs)
        next_obs, rewards, terminated, truncated, info = env.step(actions)
        done = terminated or truncated
        buffer.add(obs, actions, rewards, value, log_probs, done)
        obs = next_obs
        if done:
            obs, _ = env.reset()
    return obs

def evaluate(env, agent, n_eps):
    # eval_env has success_bonus=0, so this measures true task success.
    success = collision = 0
    for _ in range(n_eps):
        obs, _ = env.reset()
        while True:
            actions, _, _ = agent.get_actions(obs)
            obs, _, terminated, truncated, info = env.step(actions)
            if terminated or truncated:
                success   += int(info['all_targets_reached'])
                collision += int(info['any_collision'])
                break
    return success / n_eps, collision / n_eps

def save_history(history, path):
    with open(path, 'w') as f:
        json.dump(history, f, indent=2)

history = {
    'episode': [], 'success': [], 'collision': [],
    'actor_loss': [], 'critic_loss': [], 'entropy': []
}
episode      = 0
update       = 0
steps_per_ep = max(1, CFG['rollout_steps'] // CFG['max_steps'])
start        = time.time()
history_path = f"{RESULTS_DIR}/history.json"

print('Training started! (use_hungarian = False — ablation run)')
print(f'{"Episode":>8} | {"Success":>8} | {"Collision":>10} | {"A-Loss":>8} | {"Entropy":>8} | {"Time":>6}')
print('-' * 65)

while episode < CFG['total_episodes']:
    last_obs = collect_rollout(train_env, agent, buffer, CFG['rollout_steps'])
    stats    = agent.update(buffer, last_obs)
    episode += steps_per_ep
    update  += 1

    eval_interval = max(1, CFG['eval_every'] // steps_per_ep)
    if update % eval_interval == 0:
        sr, cr  = evaluate(eval_env, agent, CFG['eval_episodes'])
        elapsed = (time.time() - start) / 60
        print(f"{min(episode, CFG['total_episodes']):>8} | "
              f"{sr*100:>7.1f}% | {cr*100:>9.1f}% | "
              f"{stats['actor_loss']:>+8.4f} | "
              f"{stats['entropy']:>8.4f} | {elapsed:>5.1f}m")
        history['episode'].append(min(episode, CFG['total_episodes']))
        history['success'].append(sr)
        history['collision'].append(cr)
        history['actor_loss'].append(stats['actor_loss'])
        history['critic_loss'].append(stats['critic_loss'])
        history['entropy'].append(stats['entropy'])
        save_history(history, history_path)

    save_interval = max(1, CFG['save_every'] // steps_per_ep)
    if update % save_interval == 0:
        path = f"{RESULTS_DIR}/checkpoint_ep{min(episode, CFG['total_episodes'])}.pt"
        agent.save(path)
        print(f'  >> checkpoint saved: {path}')

sr, cr = evaluate(eval_env, agent, n_eps=50)
print(f'\n==== ABLATION TRAINING COMPLETE ====')
print(f'Final Success Rate  : {sr*100:.1f}%')
print(f'Final Collision Rate: {cr*100:.1f}%')
agent.save(f'{RESULTS_DIR}/final_model.pt')
save_history(history, history_path)
print(f'Sab kuch save ho gaya: {RESULTS_DIR}/')


In [ ]:
# Cell 6 — Results plot + comparison against v4 (Hungarian ON)
import matplotlib.pyplot as plt
import json as _json, os as _os

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['episode'], [s*100 for s in history['success']],
             color='green', linewidth=2, label='Success (Hungarian OFF)')
axes[0].plot(history['episode'], [c*100 for c in history['collision']],
             color='red', linewidth=2, linestyle='--', label='Collision (Hungarian OFF)')

# Overlay the v4 (Hungarian ON) curve if it's available locally, for a direct comparison.
v4_path = _os.path.join('..', 'v3-output', 'history.json')
if _os.path.exists(v4_path):
    with open(v4_path) as f:
        v4 = _json.load(f)
    axes[0].plot(v4['episode'], [s*100 for s in v4['success']],
                 color='green', linewidth=1, alpha=0.4, linestyle=':',
                 label='Success (Hungarian ON, v4)')
else:
    print(f'(v4 history not found at {v4_path} — plotting ablation run only)')

axes[0].set_xlabel('Episode'); axes[0].set_ylabel('%'); axes[0].set_ylim(-5, 105)
axes[0].set_title('Success / Collision — Hungarian OFF vs ON')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['episode'], history['actor_loss'], color='blue', label='Actor Loss')
axes[1].plot(history['episode'], history['critic_loss'], color='orange', label='Critic Loss')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training Losses'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['episode'], history['entropy'], color='purple', label='Entropy')
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Entropy')
axes[2].set_title('Policy Entropy'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Ablation — Hungarian Assignment OFF (fixed identity pairing)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

final_success = history['success'][-1] * 100
print(f'\nFinal success rate WITHOUT Hungarian assignment: {final_success:.1f}%')
print('Compare to v4 (WITH Hungarian assignment): 100.0%')
if final_success >= 95:
    print('=> Success stayed ~100% even without assignment optimization.')
    print('   Stage 1 as configured is TOO EASY to prove the mechanism matters.')
    print('   Next: tighten target_radius / max_steps / add obstacles before Stage 2.')
else:
    print('=> Success dropped meaningfully without assignment optimization.')
    print('   Good — this confirms the Hungarian mechanism is doing real work in Stage 1.')
